In [1]:
from astroplan.plots import plot_airmass
from astropy.coordinates import SkyCoord
from astroplan import FixedTarget
from astropy.time import Time
from astroplan import Observer
from astroplan.plots import plot_airmass
from astroplan.plots import plot_parallactic
import matplotlib.pyplot as plt
from matplotlib import dates
import numpy as n
import astropy.units as u
import etc_data
import etc_routines

from astropy.io import fits
from photutils.aperture import aperture_photometry, CircularAperture
from astropy import constants
import xlrd
#import etc_data
#import etc_routines
import analysis
from scipy.integrate import nquad
from astropy.modeling.functional_models import Gaussian2D

import pyklip.klip as klip
import astropy.stats as aps
import glob
import pyklip.rdi as rdi
import pyklip
import pyklip.instruments.GPI as GPI
import pyklip.parallelized as parallelized
from importlib import reload
import matplotlib as mpl
from spectral_cube import SpectralCube
from mpl_toolkits.axes_grid1 import make_axes_locatable
from pyklip.kpp.utils.mathfunc import *
from pyklip.kpp.metrics.crossCorr import calculate_cc
from pyklip.kpp.stat.statPerPix_utils import get_image_stat_map_perPixMasking
%matplotlib notebook

In [2]:
save_dir = 'simulated_data_R10/'
cenx = 35/2#100.5
ceny = 35/2#100.5

In [3]:
files_all = glob.glob(save_dir + 'reduced_day*/im_*.fits.gz')
#files_all = n.append(files_all, files_all)
#files_all = n.append(files_all, files_all)
#files_all = n.append(files_all, files_all)

filter_nos = n.zeros(len(files_all))

for i, filename in enumerate(files_all):
    
   filter_nos[i] = float(filename.split('_')[-2])

In [4]:
len(files_all)

267218

In [5]:
len(files_all)

267218

In [6]:
n_wavelems = len(n.unique(filter_nos))
print(n_wavelems)

7


In [7]:
collapsed_images = [None] * n_wavelems

for filter_no in n.arange(n_wavelems):
    
    print(filter_no)
    
    files = glob.glob(save_dir+'reduced_day1/im_'+str(filter_no)+'*')
    data = [None] * len(files)
    degrees = n.zeros(len(files))
    wav = n.zeros(len(files))

    for i, file_dir in enumerate(files):
        data[i] = fits.getdata(file_dir)
        #print(data[i].shape)
        degrees[i] = float(fits.getheader(file_dir)['ANG']) * -1
        wav[i] = float(fits.getheader(file_dir)['WAV'])
    
    data = n.array(data)
    
    wav0=n.round(n.mean(wav),2)
    
    n_images = data.shape[0]
    newarr = data
    subarr = n.zeros(data.shape) 
    derot = n.zeros(data.shape)
    derot_noadi = n.zeros(data.shape)

    medarr = n.nanmedian(data,axis=0)

    subarr=data-medarr
    center = (cenx, ceny)

    for i in n.arange(n_images):
    
        derot[i,:,:] = klip.rotate(subarr[i,:,:],degrees[i],center)#,new_center=(140,140))
        derot_noadi[i,:,:] = klip.rotate(newarr[i,:,:],degrees[i],center)#,new_center=(140,140))
        
    medcollapse = klip.collapse_data(derot,axis=0,collapse_method='median') 
    collapsed_images[filter_no] = medcollapse
    #fits.writeto(save_dir + 'derot_collapse.fits', medcollapse, overwrite = True)
    
    #fits.writeto(save_dir + 'derot' + str(+'.fits', derot, overwrite = True)
    #fits.writeto(save_dir + 'derot_noadi.fits', derot_noadi, overwrite = True)
    #medcollapse = klip.collapse_data(derot,axis=0,collapse_method='mean') 
    fits.writeto(save_dir + 'derot_collapse'+str(wav0)+'.fits', medcollapse, overwrite = True)
        
collapsed_images = n.array(collapsed_images)    
fits.writeto(save_dir + 'derot_collapse_all_real.fits', collapsed_images , overwrite = True)

0
1
2
3
4
5
6


In [8]:
filename = save_dir + 'derot_collapse_all_real.fits'
data = fits.getdata(filename)
center = [cenx, ceny]#[cen_x,cen_y]

SNR = [None] * n_wavelems

x_grid,y_grid= np.meshgrid(np.arange(-1,1),np.arange(-1,1))
kernel_gauss = gauss2d(x_grid,y_grid, amplitude = 1e7, xo = 0.0, yo = 0.0, sigma_x = 1, sigma_y = 1)

for i, image in enumerate(data):
    
    image_cc = calculate_cc(image, kernel_gauss,spectrum = None, nans2zero=True)
    
    SNR_map = get_image_stat_map_perPixMasking(image_cc,
                                           centroid = center,
                                           mask_radius=15,
                                           Dr = 3,
                                           type = "SNR")
    
    SNR[i] = SNR_map

SNR = n.array(SNR)
fits.writeto(save_dir + 'SNR_all.fits', SNR , overwrite = True)
    

/Users/zen/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/zen/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/zen/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/zen/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/Users/zen/miniconda3/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, d

In [9]:
SNR

array([[[        nan,         nan, -0.72308038, ..., -0.53247562,
          1.39136836,         nan],
        [        nan, -0.09318881, -2.26293065, ..., -1.63855307,
         -0.81263577, -0.65492329],
        [-0.48004484, -1.6939018 , -3.35667286, ..., -0.39134777,
         -0.09799328, -2.7324413 ],
        ...,
        [-0.60737063,  0.70386639,  0.93907279, ...,  1.1061465 ,
          0.1208531 , -0.58638856],
        [ 1.23387819,  1.10812285, -0.22479713, ..., -0.25661843,
          0.68983196, -0.09207301],
        [        nan, -1.65547158, -1.8564583 , ...,  0.60605327,
          0.92257535,  0.50820298]],

       [[        nan,         nan,  0.52110401, ..., -0.17923111,
          0.33337796,         nan],
        [        nan, -1.03445879, -0.98932461, ...,  0.95152329,
          1.29218891,  1.23385036],
        [-2.23512391, -2.09006061,  0.52870877, ...,  1.5741532 ,
          1.68285666,  1.95437797],
        ...,
        [ 0.1951132 ,  0.6076316 ,  0.43733843, ...,  

In [10]:
plt.figure('h')
plt.imshow(SNR[0], origin = 'lower')
plt.show()

<IPython.core.display.Javascript object>